In [1]:
import requests
import pandas as pd

url = "https://myhospitalsapi.aihw.gov.au/api/v1/reporting-units"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

rows = []

for ru in data["result"]:
    state = None
    hospital_type = None
    
    for mu in ru.get("mapped_reporting_units", []):
        t = mu["mapped_reporting_unit"]["reporting_unit_type"]["reporting_unit_type_name"]
        n = mu["mapped_reporting_unit"]["reporting_unit_name"]
        if t == "State":
            state = n
        if t == "Local Hospital Network":
            hospital_type = t
    
    rows.append({
        "reporting_unit_code": ru["reporting_unit_code"],
        "reporting_unit_name": ru["reporting_unit_name"],
        "state": state,
        "hospital_type": hospital_type
    })

df = pd.DataFrame(rows)
df.head()


,reporting_unit_code,reporting_unit_name,state,hospital_type
0,H0012,State Forensic Mental Health Service,Western Australia,Local Hospital Network
1,H0013,Justice Health Services,New South Wales,Local Hospital Network
2,H0014,The Children's Hospital at Westmead,New South Wales,Local Hospital Network
3,H0015,Sydney Children's Hospital,New South Wales,Local Hospital Network
4,H0016,Sacred Heart Health Service,New South Wales,Local Hospital Network


In [2]:
import requests
import pandas as pd

url = "https://myhospitalsapi.aihw.gov.au/api/v1/measures"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

rows = []

for m in data["result"]:
    rows.append({
        "measure_code": m["measure_code"],
        "measure_name": m["measure_name"],
        "description": m["measure_name"],
        "unit": m["units"]["units_name"]
    })

df = pd.DataFrame(rows)
df.head()


,measure_code,measure_name,description,unit
0,MYH0001,Number of surgeries for malignant cancer,Number of surgeries for malignant cancer,surgeries
1,MYH0002,Median waiting time for surgery for malignant ...,Median waiting time for surgery for malignant ...,days
2,MYH0003,Percentage of people who received surgery for ...,Percentage of people who received surgery for ...,percent
3,MYH0004,Percentage of people who received surgery for ...,Percentage of people who received surgery for ...,percent
4,MYH0005,Percentage of patients who depart the emergenc...,Percentage of patients who depart the emergenc...,percent


In [3]:
import requests
import pandas as pd

url = "https://myhospitalsapi.aihw.gov.au/api/v1/reported-measures"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

rows = []

for rm in data["result"]:
    rows.append({
        "measure_code": rm["measure"]["measure_code"],
        "reported_measure_code": rm["reported_measure_code"],
        "reported_measure_name": rm["reported_measure_name"]
    })

df = pd.DataFrame(rows)
df.head()


,measure_code,reported_measure_code,reported_measure_name
0,MYH0002,MYH-RM0001,Breast cancer
1,MYH0002,MYH-RM0002,Bowel cancer
2,MYH0002,MYH-RM0003,Lung cancer
3,MYH0001,MYH-RM0004,Breast cancer
4,MYH0001,MYH-RM0005,Bowel cancer


In [4]:
%pip install sqlalchemy psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


In [5]:
from sqlalchemy import create_engine
import dotenv
import os

dotenv.load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)



In [6]:
import requests
import pandas as pd

url = "https://myhospitalsapi.aihw.gov.au/api/v1/reporting-units"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

rows = []

for ru in data["result"]:
    state = None
    hospital_type = None

    for mu in ru.get("mapped_reporting_units", []):
        t = mu["mapped_reporting_unit"]["reporting_unit_type"]["reporting_unit_type_name"]
        n = mu["mapped_reporting_unit"]["reporting_unit_name"]

        if t == "State":
            state = n
        if t == "Local Hospital Network":
            hospital_type = t

    rows.append({
        "reporting_unit_code": ru["reporting_unit_code"],
        "reporting_unit_name": ru["reporting_unit_name"],
        "state": state,
        "type": hospital_type
    })

df_hospitals = pd.DataFrame(rows)

# Load into PostgreSQL
df_hospitals.to_sql(
    name="stg_hospitals",
    con=engine,
    schema="staging",
    if_exists="append",   # use "replace" only for testing
    index=False
)

print("stg_hospitals loaded successfully")


stg_hospitals loaded successfully


In [7]:
url = "https://myhospitalsapi.aihw.gov.au/api/v1/measures"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
data = res.json()

rows = []

for m in data["result"]:
    rows.append({
        "measure_code": m["measure_code"],
        "measure_name": m["measure_name"],
        "description": m["measure_name"],
        "unit": m["units"]["units_name"]
    })

df_measures = pd.DataFrame(rows)

df_measures.to_sql(
    "stg_measures",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

print("stg_measures loaded successfully")


stg_measures loaded successfully


In [8]:
url = "https://myhospitalsapi.aihw.gov.au/api/v1/reported-measures"
headers = {"User-Agent": "Mozilla/5.0"}

res = requests.get(url, headers=headers)
res.raise_for_status()
data = res.json()

# Prepare data
rows = []
for rm in data.get("result", []):
    rows.append({
      
        "measure_code": (
            rm.get("measure", {})
              .get("measure_code")
        ),
        "reported_measure_code": rm.get("reported_measure_code"),
        "reported_measure_name": rm.get("reported_measure_name"),
        "value": rm.get("value"),  # currently None
        "unit": (
            rm.get("measure", {})
              .get("units", {})
              .get("units_name")
        ),
        "data_period": rm.get("data_period")  # currently None
    })

df_reported = pd.DataFrame(rows)

# Fill dummy data if you want (optional)
# df_reported['value'] = df_reported['value'].fillna(0)
# df_reported['data_period'] = df_reported['data_period'].fillna('2025-12-31')

# Load into PostgreSQL
df_reported.to_sql(
    name="stg_reported_measure_values",
    con=engine,
    schema="staging",
    if_exists="replace",  # drops table if exists and creates new one
    index=False,
    method="multi"
)

print("Data loaded successfully!")
print(df_reported.head())

Data loaded successfully!
  measure_code reported_measure_code reported_measure_name value       unit  \
0      MYH0002            MYH-RM0001         Breast cancer  None       days   
1      MYH0002            MYH-RM0002          Bowel cancer  None       days   
2      MYH0002            MYH-RM0003           Lung cancer  None       days   
3      MYH0001            MYH-RM0004         Breast cancer  None  surgeries   
4      MYH0001            MYH-RM0005          Bowel cancer  None  surgeries   

  data_period  
0        None  
1        None  
2        None  
3        None  
4        None  


In [9]:
import pandas as pd
import random
from sqlalchemy import create_engine, text



# Load existing table
df = pd.read_sql("SELECT * FROM staging.stg_reported_measure_values", engine)

# Function to generate realistic dummy values
def generate_value(unit):
    if unit is None:
        return None
    unit = unit.lower()
    if "percent" in unit:
        return round(random.uniform(50, 100), 2)
    if "day" in unit:
        return random.randint(1, 120)
    if "minute" in unit:
        return random.randint(10, 300)
    if "dollar" in unit:
        return round(random.uniform(1000, 50000), 2)
    if "patient" in unit or "case" in unit or "surgery" in unit:
        return random.randint(0, 5000)
    return random.randint(1, 100)

# Fill only rows where value or data_period is NULL
df['value'] = df.apply(lambda x: generate_value(x['unit']) if pd.isna(x['value']) else x['value'], axis=1)
df['data_period'] = df['data_period'].fillna("2022-23")  # dummy period

# Use a connection to execute updates
with engine.connect() as conn:
    for _, row in df.iterrows():
        conn.execute(
            text("""
                UPDATE staging.stg_reported_measure_values
                SET value = :value,
                    data_period = :data_period
                WHERE measure_code = :measure_code
                  AND reported_measure_code = :reported_measure_code
                  AND (value IS NULL OR data_period IS NULL)
            """),
            {
                "value": row['value'],
                "data_period": row['data_period'],
                "measure_code": row['measure_code'],
                "reported_measure_code": row['reported_measure_code']
            }
        )
    conn.commit()  # commit the transaction

print("NULL values filled with dummy data successfully!")


NULL values filled with dummy data successfully!
